# Qwen Family
Environment: envs/qwen.yaml

In [ ]:
import subprocess
import sys
import re
import json
import time
from pathlib import Path

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/content/LLMComparison'),
    Path('/content/drive/MyDrive/LLMComparison'),
]

PROJECT_ROOT = next(
    (
        root
        for root in candidate_roots
        if (root / 'experiments').exists() and (root / 'src').exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError('Project root not found.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# === Qwen Family Configuration ===
MODELS = ['qwen2-vl-2b', 'qwen2.5-vl-3b', 'qwen3-vl-2b']
PRESET = 'free_colab_t4'
DATASETS = ['hf_vqa_rad']
NUM_SAMPLES = 30
SEED = 42
MAX_FALLBACK_RATE = 1.0
STRICT_SAMPLE_VALIDATION = False
OUTPUT_DIR = PROJECT_ROOT / 'results'
RUN_NAME = 'qwen_family_vqa'

print(f'Qwen Family Benchmark')
print(f'Models: {", ".join(MODELS)}')
print(f'Preset: {PRESET}')
print(f'Run name: {RUN_NAME}')
print(f'Seed: {SEED}')
print(f'Max fallback rate: {MAX_FALLBACK_RATE}')

command = [
    sys.executable,
    str(PROJECT_ROOT / 'experiments' / 'run_unified.py'),
    '--preset', PRESET,
    '--models', *MODELS,
    '--datasets', *DATASETS,
    '--num-samples', str(NUM_SAMPLES),
    '--seed', str(SEED),
    '--max-fallback-rate', str(MAX_FALLBACK_RATE),
    '--skip-inaccessible',
    '--output-dir', str(OUTPUT_DIR),
    '--run-name', RUN_NAME,
]

if STRICT_SAMPLE_VALIDATION:
    command.append('--strict-sample-validation')

print('\nRunning multi-model benchmark...')

completed_samples = 0
total_samples = None
start_ts = time.time()

process = subprocess.Popen(
    command,
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    text_line = line.rstrip()

    sample_match = re.search(r'Progress model=.* sample=(\d+)/(\d+)', text_line)
    if sample_match:
        completed_samples = int(sample_match.group(1))
        total_samples = int(sample_match.group(2))

    elapsed = time.time() - start_ts
    elapsed_min = elapsed / 60.0

    if total_samples and completed_samples > 0:
        avg_per_sample = elapsed / completed_samples
        remaining_samples = max(0, total_samples - completed_samples)
        eta_sec = avg_per_sample * remaining_samples
        eta_min = eta_sec / 60.0
        print(
            f'[sample {completed_samples}/{total_samples}] elapsed={elapsed_min:.1f}m eta~{eta_min:.1f}m | {text_line}'
        )
    else:
        print(f'[sample ?/?] elapsed={elapsed_min:.1f}m eta~unknown | {text_line}')

return_code = process.wait()
total_elapsed_min = (time.time() - start_ts) / 60.0
print(f'\nFinished with code={return_code} in {total_elapsed_min:.1f}m')

if return_code != 0:
    raise RuntimeError(f'run_unified failed with exit code {return_code}')

# === Results Summary ===
import pandas as pd

run_dir = OUTPUT_DIR / RUN_NAME
aggregate_path = run_dir / 'aggregate_metrics.json'
stats_path = run_dir / 'stats.json'
errors_path = run_dir / 'errors.jsonl'

print('\n=== RESULTS SUMMARY ===')

if aggregate_path.exists():
    aggregate_payload = json.loads(aggregate_path.read_text())
    aggregate_rows = [
        {'model_name': model_name, **metrics}
        for model_name, metrics in aggregate_payload.get('metrics', {}).items()
    ]
    aggregate_df = pd.DataFrame(aggregate_rows)
    print('\nAggregate metrics (Qwen Family):')
    display(aggregate_df.T if not aggregate_df.empty else aggregate_df)

    fallback_columns = [c for c in aggregate_df.columns if c.endswith('_fallback_used_mean')]
    if fallback_columns:
        print('\nFallback usage summary:')
        display(aggregate_df[['model_name', *fallback_columns]])

if stats_path.exists():
    stats_payload = json.loads(stats_path.read_text())
    fallback_rates = stats_payload.get('statistics', {}).get('_meta_fallback_rates', {})
    if fallback_rates:
        print('\nFallback rates (metadata):')
        display(pd.DataFrame([fallback_rates]))

if errors_path.exists():
    error_rows = [json.loads(line) for line in errors_path.read_text().splitlines() if line.strip()]
    if error_rows:
        error_df = pd.DataFrame(error_rows)
        print('\nError type counts:')
        display(error_df.groupby('error_type').size().to_frame('count').T)